In [3]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [4]:
!pip install -q segmentation-models-pytorch albumentations opencv-python-headless \
    scikit-image matplotlib pandas numpy pillow scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.6 MB/s eta 0:00:00


In [5]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
import segmentation_models_pytorch as smp

from sklearn.model_selection import train_test_split

print("Libraries imported successfully!")
print("SMP version:", smp.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Libraries imported successfully!
SMP version: 0.5.0
GPU: Tesla T4


In [6]:
PROJECT_DIR = "/content/Liver_Fibrosis_Segmentation"

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/data/images", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/data/masks", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/models", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/outputs", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/notebooks", exist_ok=True)

print("Project structure created!")

Project structure created!


In [7]:
!find /content/Liver_Fibrosis_Segmentation -maxdepth 2 -type d

/content/Liver_Fibrosis_Segmentation
/content/Liver_Fibrosis_Segmentation/notebooks
/content/Liver_Fibrosis_Segmentation/models
/content/Liver_Fibrosis_Segmentation/outputs
/content/Liver_Fibrosis_Segmentation/data
/content/Liver_Fibrosis_Segmentation/data/images
/content/Liver_Fibrosis_Segmentation/data/masks


In [8]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)

Using device: cuda


In [9]:
!pip install -q requests

In [10]:
import requests

url = "https://zenodo.org/records/8429531"

response = requests.get(url)

print("Status:", response.status_code)
print("Page length:", len(response.text))

Status: 200
Page length: 73727


In [11]:
import requests
import re

html = response.text

# Find downloadable file links
links = re.findall(r'https?://[^"\']+', html)

for link in links:
    if any(x in link.lower() for x in [".zip", ".tar", ".gz", ".png", ".jpg", ".tif"]):
        print(link)

In [12]:
import requests

api_url = "https://zenodo.org/api/records/8429531"

data = requests.get(api_url).json()

print("Dataset title:")
print(data["metadata"]["title"])

print("\nAvailable files:")
for file in data["files"]:
    print("Name:", file["key"])
    print("Size:", file["size"], "bytes")
    print("Download URL:", file["links"]["self"])
    print("-" * 80)

Dataset title:
Single-cell and spatially resolved transcriptomic data of mouse regenerative livers under normal and fibrotic conditions

Available files:
Name: ST-HE-Fibrosis-1mm.7z
Size: 1210814921 bytes
Download URL: https://zenodo.org/api/records/8429531/files/ST-HE-Fibrosis-1mm.7z/content
--------------------------------------------------------------------------------
Name: ST-HE-Normal-1mm.7z
Size: 1406551328 bytes
Download URL: https://zenodo.org/api/records/8429531/files/ST-HE-Normal-1mm.7z/content
--------------------------------------------------------------------------------
Name: ST-ssDNA-Fibrosis.7z
Size: 5995310056 bytes
Download URL: https://zenodo.org/api/records/8429531/files/ST-ssDNA-Fibrosis.7z/content
--------------------------------------------------------------------------------
Name: ST-ssDNA-Normal.7z
Size: 5229351444 bytes
Download URL: https://zenodo.org/api/records/8429531/files/ST-ssDNA-Normal.7z/content
-------------------------------------------------------

In [13]:
import os
import requests
from tqdm.auto import tqdm

PROJECT_DIR = "/content/Liver_Fibrosis_Segmentation"
DATA_DIR = f"{PROJECT_DIR}/data"

os.makedirs(DATA_DIR, exist_ok=True)

fibrosis_url = "https://zenodo.org/api/records/8429531/files/ST-HE-Fibrosis-1mm.7z/content"

fibrosis_path = f"{DATA_DIR}/ST-HE-Fibrosis-1mm.7z"

response = requests.get(fibrosis_url, stream=True)
response.raise_for_status()

total_size = int(response.headers.get("content-length", 0))

with open(fibrosis_path, "wb") as f:
    with tqdm(
        total=total_size,
        unit="B",
        unit_scale=True,
        desc="Downloading Fibrosis H&E"
    ) as pbar:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                pbar.update(len(chunk))

print("Download completed!")
print("Saved to:", fibrosis_path)

Download completed!
Saved to: /content/Liver_Fibrosis_Segmentation/data/ST-HE-Fibrosis-1mm.7z


In [14]:
import os

file_path = "/content/Liver_Fibrosis_Segmentation/data/ST-HE-Fibrosis-1mm.7z"

print("File exists:", os.path.exists(file_path))
print("File size (MB):", round(os.path.getsize(file_path) / (1024**2), 2))

File exists: True
File size (MB): 1154.72


In [15]:
!apt-get update -qq
!apt-get install -y p7zip-full -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [16]:
!7z x "/content/Liver_Fibrosis_Segmentation/data/ST-HE-Fibrosis-1mm.7z" \
-o"/content/Liver_Fibrosis_Segmentation/data/fibrosis"


7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:2 OPEN_MAX:1048576

Scanning the drive for archives:
  0M Scan /content/Liver_Fibrosis_Segmentation/data/                                                    1 file, 1210814921 bytes (1155 MiB)

Extracting archive: /content/Liver_Fibrosis_Segmentation/data/ST-HE-Fibrosis-1mm.7z
--
Path = /content/Liver_Fibrosis_Segmentation/data/ST-HE-Fibrosis-1mm.7z
Type = 7z
Physical Size = 1210814921
Headers Size = 400
Method = LZMA2:24
Solid = +
Blocks = 1

  0%      1%      2%      3%      4%      5%      6%      7%      8%      9%      9% - F01.tif                9% 1 - F02.tif                 10% 1 - F02.tif                

In [17]:
# ============================================================
# LIVER FIBROSIS SEGMENTATION - DATASET SETUP
# ============================================================

import os
import requests
import subprocess
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. PROJECT DIRECTORIES
# ------------------------------------------------------------

PROJECT_DIR = "/content/Liver_Fibrosis_Segmentation"

DATA_DIR = f"{PROJECT_DIR}/data"
IMAGE_DIR = f"{DATA_DIR}/images"
MASK_DIR = f"{DATA_DIR}/masks"
MODEL_DIR = f"{PROJECT_DIR}/models"
OUTPUT_DIR = f"{PROJECT_DIR}/outputs"
NOTEBOOK_DIR = f"{PROJECT_DIR}/notebooks"

for folder in [
    PROJECT_DIR,
    DATA_DIR,
    IMAGE_DIR,
    MASK_DIR,
    MODEL_DIR,
    OUTPUT_DIR,
    NOTEBOOK_DIR
]:
    os.makedirs(folder, exist_ok=True)

print("Project folders created!")
print(PROJECT_DIR)


# ------------------------------------------------------------
# 2. DATASET URL
# ------------------------------------------------------------

fibrosis_url = (
    "https://zenodo.org/api/records/8429531/"
    "files/ST-HE-Fibrosis-1mm.7z/content"
)

archive_path = f"{DATA_DIR}/ST-HE-Fibrosis-1mm.7z"
extract_dir = f"{DATA_DIR}/fibrosis"


# ------------------------------------------------------------
# 3. DOWNLOAD DATASET
# ------------------------------------------------------------

if os.path.exists(archive_path):

    file_size = os.path.getsize(archive_path) / (1024 ** 2)

    print(f"Dataset already exists.")
    print(f"Current size: {file_size:.2f} MB")

else:

    print("Starting dataset download...")

    response = requests.get(
        fibrosis_url,
        stream=True,
        timeout=120
    )

    response.raise_for_status()

    total_size = int(
        response.headers.get("content-length", 0)
    )

    with open(archive_path, "wb") as f:

        with tqdm(
            total=total_size,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc="Downloading Fibrosis H&E"
        ) as pbar:

            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):

                if chunk:

                    f.write(chunk)
                    pbar.update(len(chunk))

    print("\nDownload completed!")


# ------------------------------------------------------------
# 4. CHECK DOWNLOADED FILE
# ------------------------------------------------------------

print("\n===== DATASET FILE =====")

if os.path.exists(archive_path):

    size_mb = os.path.getsize(archive_path) / (1024 ** 2)

    print("File exists     :", True)
    print("File size (MB)  :", round(size_mb, 2))
    print("File path       :", archive_path)

else:

    raise FileNotFoundError(
        "Dataset download failed!"
    )


# ------------------------------------------------------------
# 5. INSTALL 7-ZIP
# ------------------------------------------------------------

print("\nInstalling 7-Zip...")

subprocess.run(
    [
        "apt-get",
        "update",
        "-qq"
    ],
    check=True
)

subprocess.run(
    [
        "apt-get",
        "install",
        "-y",
        "p7zip-full",
        "-qq"
    ],
    check=True
)

print("7-Zip installed!")


# ------------------------------------------------------------
# 6. EXTRACT DATASET
# ------------------------------------------------------------

os.makedirs(extract_dir, exist_ok=True)

print("\nExtracting dataset...")
print("This may take some time...")

result = subprocess.run(
    [
        "7z",
        "x",
        archive_path,
        f"-o{extract_dir}",
        "-y"
    ],
    capture_output=True,
    text=True
)

print(result.stdout[-3000:])

if result.returncode != 0:

    print(result.stderr)

    raise RuntimeError(
        "Extraction failed!"
    )

print("\nExtraction completed!")


# ------------------------------------------------------------
# 7. INSPECT EXTRACTED DATA
# ------------------------------------------------------------

print("\n========================================")
print("EXTRACTED DATASET STRUCTURE")
print("========================================")

file_count = 0

for root, dirs, files in os.walk(extract_dir):

    level = root.replace(
        extract_dir, ""
    ).count(os.sep)

    indent = "  " * level

    print(
        f"{indent}{os.path.basename(root)}/"
    )

    for file in files[:10]:

        print(
            f"{indent}  {file}"
        )

    file_count += len(files)

print("\nTotal extracted files:", file_count)


# ------------------------------------------------------------
# 8. COUNT FILE TYPES
# ------------------------------------------------------------

from collections import Counter

extensions = Counter()

for root, dirs, files in os.walk(extract_dir):

    for file in files:

        ext = os.path.splitext(file)[1].lower()

        if ext:
            extensions[ext] += 1

print("\n========================================")
print("FILE TYPES")
print("========================================")

for ext, count in extensions.most_common():

    print(
        f"{ext:10s} : {count}"
    )


# ------------------------------------------------------------
# 9. SHOW FIRST 30 FILES
# ------------------------------------------------------------

print("\n========================================")
print("FIRST 30 FILES")
print("========================================")

shown = 0

for root, dirs, files in os.walk(extract_dir):

    for file in files:

        print(
            os.path.join(root, file)
        )

        shown += 1

        if shown >= 30:
            break

    if shown >= 30:
        break


# ------------------------------------------------------------
# 10. FINAL STATUS
# ------------------------------------------------------------

print("\n========================================")
print("SETUP COMPLETED")
print("========================================")

print("Project directory :", PROJECT_DIR)
print("Dataset archive   :", archive_path)
print("Extracted data    :", extract_dir)
print("Total files       :", file_count)

print("\nNext step:")
print("We will inspect the extracted images/annotations")
print("and create the image-mask segmentation dataset.")

Project folders created!
/content/Liver_Fibrosis_Segmentation
Dataset already exists.
Current size: 1154.72 MB

===== DATASET FILE =====
File exists     : True
File size (MB)  : 1154.72
File path       : /content/Liver_Fibrosis_Segmentation/data/ST-HE-Fibrosis-1mm.7z

Installing 7-Zip...
7-Zip installed!

Extracting dataset...
This may take some time...

7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:2 OPEN_MAX:1048576

Scanning the drive for archives:
1 file, 1210814921 bytes (1155 MiB)

Extracting archive: /content/Liver_Fibrosis_Segmentation/data/ST-HE-Fibrosis-1mm.7z
--
Path = /content/Liver_Fibrosis_Segmentation/data/ST-HE-Fibrosis-1mm.7z
Type = 7z
Physical Size = 1210814921
Headers Size = 400
Method = LZMA2:24
Solid = +
Blocks = 1

Everything is Ok

Files: 15
Size:       1418983244
Compressed: 1210814921


Extraction completed!

EXTRACTED DATASET STRUCTURE
fibrosis/
  F08.tif
  F06.tif
  F05.tif
  F04.tif
  F09.tif
  F01.ti

In [18]:
import os

EXTRACT_DIR = "/content/Liver_Fibrosis_Segmentation/data/fibrosis"

print("=" * 70)
print("ALL EXTRACTED FILES")
print("=" * 70)

all_files = []

for root, dirs, files in os.walk(EXTRACT_DIR):
    for file in files:
        full_path = os.path.join(root, file)
        size_mb = os.path.getsize(full_path) / (1024 ** 2)

        all_files.append((full_path, size_mb))

for i, (path, size) in enumerate(all_files, 1):
    print(f"{i:02d}. {path}")
    print(f"    Size: {size:.2f} MB")

print("\n" + "=" * 70)
print("TOTAL FILES:", len(all_files))
print("=" * 70)

ALL EXTRACTED FILES
01. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F08.tif
    Size: 89.16 MB
02. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F06.tif
    Size: 78.31 MB
03. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F05.tif
    Size: 98.64 MB
04. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F04.tif
    Size: 75.00 MB
05. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F09.tif
    Size: 76.02 MB
06. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F01.tif
    Size: 78.91 MB
07. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F15.tif
    Size: 108.32 MB
08. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F12.tif
    Size: 90.93 MB
09. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F02.tif
    Size: 61.57 MB
10. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F13.tif
    Size: 107.38 MB
11. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F14.tif
    Size: 124.12 MB
12. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F10.tif
    Size: 86

In [19]:
import os

EXTRACT_DIR = "/content/Liver_Fibrosis_Segmentation/data/fibrosis"

print("=" * 70)
print("ALL EXTRACTED FILES")
print("=" * 70)

all_files = []

for root, dirs, files in os.walk(EXTRACT_DIR):
    for file in files:
        full_path = os.path.join(root, file)
        size_mb = os.path.getsize(full_path) / (1024 ** 2)

        all_files.append((full_path, size_mb))

for i, (path, size) in enumerate(all_files, 1):
    print(f"{i:02d}. {path}")
    print(f"    Size: {size:.2f} MB")

print("\n" + "=" * 70)
print("TOTAL FILES:", len(all_files))
print("=" * 70)

ALL EXTRACTED FILES
01. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F08.tif
    Size: 89.16 MB
02. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F06.tif
    Size: 78.31 MB
03. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F05.tif
    Size: 98.64 MB
04. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F04.tif
    Size: 75.00 MB
05. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F09.tif
    Size: 76.02 MB
06. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F01.tif
    Size: 78.91 MB
07. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F15.tif
    Size: 108.32 MB
08. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F12.tif
    Size: 90.93 MB
09. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F02.tif
    Size: 61.57 MB
10. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F13.tif
    Size: 107.38 MB
11. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F14.tif
    Size: 124.12 MB
12. /content/Liver_Fibrosis_Segmentation/data/fibrosis/F10.tif
    Size: 86

In [20]:
from collections import Counter
import os

extensions = Counter()

for path, size in all_files:
    ext = os.path.splitext(path)[1].lower()
    extensions[ext if ext else "[no extension]"] += 1

print("File type summary:")
for ext, count in extensions.items():
    print(f"{ext}: {count}")

File type summary:
.tif: 15


In [21]:
import os
import tifffile
import numpy as np

FIBROSIS_DIR = "/content/Liver_Fibrosis_Segmentation/data/fibrosis"

tif_files = sorted([
    os.path.join(FIBROSIS_DIR, f)
    for f in os.listdir(FIBROSIS_DIR)
    if f.lower().endswith((".tif", ".tiff"))
])

print("=" * 80)
print("TIFF DATASET INSPECTION")
print("=" * 80)

for path in tif_files:

    print("\n" + "-" * 80)
    print("FILE:", os.path.basename(path))

    try:
        with tifffile.TiffFile(path) as tif:

            print("Number of pages:", len(tif.pages))
            print("Number of series:", len(tif.series))

            for i, series in enumerate(tif.series):

                print(f"\n  Series {i}")
                print("  Shape :", series.shape)
                print("  Axes  :", series.axes)
                print("  Dtype :", series.dtype)

            # Metadata
            print("\n  Metadata:")
            print("  ImageJ:", tif.is_imagej)
            print("  OME   :", tif.ome_metadata is not None)

    except Exception as e:

        print("ERROR:", e)

print("\n" + "=" * 80)
print("TOTAL TIFF FILES:", len(tif_files))
print("=" * 80)

TIFF DATASET INSPECTION

--------------------------------------------------------------------------------
FILE: F01.tif
Number of pages: 1
Number of series: 1

  Series 0
  Shape : (19917, 33312, 3)
  Axes  : YXS
  Dtype : uint8

  Metadata:
  ImageJ: False
  OME   : False

--------------------------------------------------------------------------------
FILE: F02.tif
Number of pages: 1
Number of series: 1

  Series 0
  Shape : (17745, 32416, 3)
  Axes  : YXS
  Dtype : uint8

  Metadata:
  ImageJ: False
  OME   : False

--------------------------------------------------------------------------------
FILE: F03.tif
Number of pages: 1
Number of series: 1

  Series 0
  Shape : (18767, 40548, 3)
  Axes  : YXS
  Dtype : uint8

  Metadata:
  ImageJ: False
  OME   : False

--------------------------------------------------------------------------------
FILE: F04.tif
Number of pages: 1
Number of series: 1

  Series 0
  Shape : (18353, 34518, 3)
  Axes  : YXS
  Dtype : uint8

  Metadata:
  ImageJ

In [22]:
import os
import tifffile
import matplotlib.pyplot as plt

THUMB_DIR = "/content/Liver_Fibrosis_Segmentation/outputs/thumbnails"
os.makedirs(THUMB_DIR, exist_ok=True)

print("Creating thumbnails...")

for path in tif_files:

    filename = os.path.splitext(os.path.basename(path))[0]

    try:
        with tifffile.TiffFile(path) as tif:

            series = tif.series[0]

            # Read first series
            image = series.asarray()

            print(
                filename,
                "->",
                image.shape
            )

            # Handle RGB / channel ordering
            if image.ndim == 3:

                # If channels are first
                if image.shape[0] <= 4:
                    image = image.transpose(1, 2, 0)

                # Keep only RGB if more channels exist
                if image.shape[-1] > 3:
                    image = image[..., :3]

            plt.figure(figsize=(6, 6))
            plt.imshow(image)
            plt.title(filename)
            plt.axis("off")
            plt.tight_layout()

            output_path = os.path.join(
                THUMB_DIR,
                filename + ".png"
            )

            plt.savefig(
                output_path,
                dpi=120,
                bbox_inches="tight"
            )

            plt.close()

    except Exception as e:

        print(
            "Could not create thumbnail for",
            filename,
            ":",
            e
        )

print("\nThumbnail generation completed!")
print("Saved to:", THUMB_DIR)

Creating thumbnails...
Could not create thumbnail for F01 : <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
Could not create thumbnail for F02 : <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
Could not create thumbnail for F03 : <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
Could not create thumbnail for F04 : <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
Could not create thumbnail for F05 : <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
Could not create thumbnail for F06 : <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
Could not create thumbnail for F07 : <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
Could not create thumbnail for F08 : <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
Could not create thumbnail for F09 : <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
Could not create thumbnail for F10 : <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
Could not create thumbnail for F11 : 

In [23]:
from IPython.display import display, Image
import os

for file in sorted(os.listdir(THUMB_DIR)):

    if file.endswith(".png"):

        print(file)

        display(
            Image(
                filename=os.path.join(
                    THUMB_DIR,
                    file
                )
            )
        )

In [24]:
!pip install -q imagecodecs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 77.7 MB/s eta 0:00:00


In [25]:
import os
import tifffile
import matplotlib.pyplot as plt

FIBROSIS_DIR = "/content/Liver_Fibrosis_Segmentation/data/fibrosis"
THUMB_DIR = "/content/Liver_Fibrosis_Segmentation/outputs/thumbnails"

os.makedirs(THUMB_DIR, exist_ok=True)

tif_files = sorted([
    os.path.join(FIBROSIS_DIR, f)
    for f in os.listdir(FIBROSIS_DIR)
    if f.lower().endswith((".tif", ".tiff"))
])

print("Creating thumbnails...")

for path in tif_files:

    filename = os.path.splitext(os.path.basename(path))[0]

    try:
        image = tifffile.imread(path)

        print(filename, "->", image.shape, image.dtype)

        # Convert CHW → HWC if required
        if image.ndim == 3 and image.shape[0] <= 4:
            image = image.transpose(1, 2, 0)

        # Keep RGB
        if image.ndim == 3 and image.shape[-1] > 3:
            image = image[..., :3]

        plt.figure(figsize=(7, 7))
        plt.imshow(image)
        plt.title(filename)
        plt.axis("off")
        plt.tight_layout()

        output_path = os.path.join(
            THUMB_DIR,
            filename + ".png"
        )

        plt.savefig(
            output_path,
            dpi=100,
            bbox_inches="tight"
        )

        plt.close()

    except Exception as e:

        print(f"ERROR for {filename}: {e}")

print("\nThumbnail generation completed!")
print("Saved to:", THUMB_DIR)

Creating thumbnails...
ERROR for F01: <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
ERROR for F02: <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
ERROR for F03: <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
ERROR for F04: <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
ERROR for F05: <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
ERROR for F06: <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
ERROR for F07: <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
ERROR for F08: <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
ERROR for F09: <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
ERROR for F10: <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
ERROR for F11: <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
ERROR for F12: <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
ERROR for F13: <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package
ERROR for F14: <COMPRESSION.JPEG: 7> requ

In [26]:
!pip install -q imagecodecs

In [27]:
import tifffile
import matplotlib.pyplot as plt

img_path = "/content/Liver_Fibrosis_Segmentation/data/fibrosis/F01.tif"

img = tifffile.imread(img_path)

print("Image shape:", img.shape)
print("Image dtype:", img.dtype)

plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.axis("off")
plt.title("F01 Liver Fibrosis Image")
plt.show()

ValueError: <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package

In [28]:
!pip install -U imagecodecs

In [29]:
import imagecodecs
import tifffile

print("imagecodecs:", imagecodecs.__version__)
print("tifffile:", tifffile.__version__)

imagecodecs: 2026.8.16
tifffile: 2026.9.15


In [30]:
import tifffile
import matplotlib.pyplot as plt

img_path = "/content/Liver_Fibrosis_Segmentation/data/fibrosis/F01.tif"

img = tifffile.imread(img_path)

print("Image shape:", img.shape)
print("Image dtype:", img.dtype)

ValueError: <COMPRESSION.JPEG: 7> requires the 'imagecodecs' package

In [31]:
import imagecodecs
import tifffile

print("imagecodecs version:", imagecodecs.__version__)
print("JPEG decoder available:", hasattr(imagecodecs, "jpeg_decode"))
print("JPEG encoder available:", hasattr(imagecodecs, "jpeg_encode"))

print("tifffile version:", tifffile.__version__)

imagecodecs version: 2026.8.16
JPEG decoder available: True
JPEG encoder available: True
tifffile version: 2026.9.15


In [32]:
import imagecodecs

print(imagecodecs.jpeg_decode)

<function jpeg_decode at 0x7824e5717ce0>


In [33]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

img_path = "/content/Liver_Fibrosis_Segmentation/data/fibrosis/F01.tif"

pil_img = Image.open(img_path)

print("PIL format:", pil_img.format)
print("PIL size:", pil_img.size)
print("PIL mode:", pil_img.mode)

img = np.array(pil_img)

print("Image shape:", img.shape)
print("Image dtype:", img.dtype)

DecompressionBombError: Image size (663475104 pixels) exceeds limit of 178956970 pixels, could be decompression bomb DOS attack.

In [1]:
import os

PROJECT_DIR = "/content/Liver_Fibrosis_Segmentation"

folders = [
    "notebooks",
    "src",
    "models",
    "outputs/figures",
    "outputs/predictions",
    "outputs/metrics",
    "data"
]

for folder in folders:
    os.makedirs(os.path.join(PROJECT_DIR, folder), exist_ok=True)

# Create placeholder files
files = [
    "src/__init__.py",
    "models/.gitkeep",
    "outputs/figures/.gitkeep",
    "outputs/predictions/.gitkeep",
    "outputs/metrics/.gitkeep",
]

for file in files:
    path = os.path.join(PROJECT_DIR, file)
    open(path, "w").close()

print("GitHub project structure created!")

GitHub project structure created!


In [2]:
requirements = """torch
torchvision
segmentation-models-pytorch
numpy
pandas
opencv-python
Pillow
matplotlib
scikit-learn
tqdm
imagecodecs
tifffile
albumentations
"""

with open(f"{PROJECT_DIR}/requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt created")

requirements.txt created


In [3]:
gitignore = """# Python
__pycache__/
*.py[cod]
*.pyo

# Jupyter
.ipynb_checkpoints/

# Virtual environments
venv/
env/
.venv/

# Dataset
data/fibrosis/
data/*.7z
data/*.zip
data/*.tar
data/*.gz

# Model weights
models/*.pth
models/*.pt
models/*.ckpt

# Outputs
outputs/predictions/*
outputs/figures/*
outputs/metrics/*

# OS
.DS_Store
Thumbs.db

# Colab
/content/
"""

with open(f"{PROJECT_DIR}/.gitignore", "w") as f:
    f.write(gitignore)

print(".gitignore created")

.gitignore created


In [4]:
data_readme = """# Dataset

This project uses the Liver Fibrosis dataset containing H&E histology images.

The original dataset is downloaded from the Zenodo repository:

https://zenodo.org/records/8429531

The dataset files are intentionally NOT included in this GitHub repository because
the source TIFF files are very large.

Expected local structure:

data/
└── fibrosis/
    ├── F01.tif
    ├── F02.tif
    ├── F03.tif
    ├── ...
    └── F15.tif

Download the required dataset from the original source before running the notebook.
"""

with open(f"{PROJECT_DIR}/data/README.md", "w") as f:
    f.write(data_readme)

print("Dataset README created")

Dataset README created


In [5]:
data_loader_code = '''from PIL import Image
import numpy as np


def load_image(path):
    """
    Load a TIFF histology image and convert it to RGB NumPy array.
    """
    image = Image.open(path).convert("RGB")
    return np.array(image)


def get_image_info(path):
    """
    Return basic information about an image.
    """
    image = Image.open(path)

    return {
        "path": path,
        "format": image.format,
        "size": image.size,
        "mode": image.mode
    }
'''

with open(f"{PROJECT_DIR}/src/data_loader.py", "w") as f:
    f.write(data_loader_code)

print("data_loader.py created")

data_loader.py created


In [6]:
preprocessing_code = '''import numpy as np
from PIL import Image


def resize_image(image, size=(512, 512)):
    """
    Resize an RGB image.
    """
    pil_image = Image.fromarray(image)
    pil_image = pil_image.resize(size)
    return np.array(pil_image)


def normalize_image(image):
    """
    Normalize pixel values from [0, 255] to [0, 1].
    """
    image = image.astype(np.float32) / 255.0
    return image
'''

with open(f"{PROJECT_DIR}/src/preprocessing.py", "w") as f:
    f.write(preprocessing_code)

print("preprocessing.py created")

preprocessing.py created


In [7]:
dataset_code = '''import torch
from torch.utils.data import Dataset


class LiverFibrosisDataset(Dataset):

    def __init__(self, images, masks=None, transform=None):
        self.images = images
        self.masks = masks
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        image = self.images[idx]

        if self.transform:
            image = self.transform(image)

        if self.masks is not None:
            mask = self.masks[idx]
            return image, mask

        return image
'''

with open(f"{PROJECT_DIR}/src/dataset.py", "w") as f:
    f.write(dataset_code)

print("dataset.py created")

dataset.py created


In [8]:
model_code = '''import segmentation_models_pytorch as smp


def create_model(
    architecture="DeepLabV3Plus",
    encoder="resnet34",
    classes=2
):

    if architecture == "DeepLabV3Plus":

        model = smp.DeepLabV3Plus(
            encoder_name=encoder,
            encoder_weights="imagenet",
            in_channels=3,
            classes=classes
        )

        return model

    raise ValueError(f"Unsupported architecture: {architecture}")
'''

with open(f"{PROJECT_DIR}/src/model.py", "w") as f:
    f.write(model_code)

print("model.py created")

model.py created


In [9]:
readme = """# Liver Fibrosis Segmentation

Deep learning based semantic segmentation of liver fibrosis regions from H&E histopathology images.

## Project Status

🚧 Currently under development.

The current stage includes:

- Dataset acquisition
- Dataset extraction
- TIFF image inspection
- Image loading
- Initial preprocessing setup
- Segmentation pipeline design

Model training and final evaluation are currently being developed.

---

## Objective

The objective of this project is to automatically identify and segment fibrosis-related regions in liver histopathology images.

The problem is formulated as a semantic segmentation task:

Input:
H&E histopathology image

Output:
Pixel-level segmentation mask

---

## Dataset

The project uses the Liver Fibrosis dataset available through Zenodo:

https://zenodo.org/records/8429531

The dataset contains H&E liver tissue images under normal and fibrotic conditions.

The original dataset is NOT included in this repository because of its large size.

---

## Tech Stack

- Python
- PyTorch
- Segmentation Models PyTorch
- DeepLabV3+
- NumPy
- Pillow
- OpenCV
- Matplotlib
- Scikit-learn
- Albumentations
- ImageCodecs
- Jupyter / Google Colab

---

## Pipeline

```text
Dataset
   ↓
TIFF Image Loading
   ↓
Image Inspection
   ↓
Preprocessing
   ↓
Annotation / Mask Preparation
   ↓
Patch Generation
   ↓
Train / Validation / Test Split
   ↓
DataLoader
   ↓
DeepLabV3+
   ↓
Training
   ↓
Validation
   ↓
Dice / IoU Evaluation
   ↓
Prediction Visualization

_IncompleteInputError: incomplete input (2646070460.py, line 1)